# Initialization





In [22]:
import pandas as pd


df = pd.read_csv('club_signups.csv')
df.head()

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


In [23]:
print('DataFrame shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)

for column in df.columns:
    print(f'\nValue counts for {column}:')
    print(df[column].value_counts(dropna=False))

DataFrame shape: (39, 9)

Column dtypes:
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

Value counts for signup_id:
signup_id
S015    2
S022    2
S023    2
S028    1
S017    1
S031    1
S014    1
S011    1
S018    1
S030    1
S016    1
S029    1
S034    1
S009    1
S021    1
S013    1
S032    1
S019    1
S024    1
S010    1
S020    1
S008    1
S035    1
S004    1
S012    1
S007    1
S005    1
S033    1
S001    1
S003    1
S006    1
S026    1
S025    1
S002    1
S027    1
S036    1
Name: count, dtype: int64

Value counts for student_id:
student_id
61-7344    6
55-9245    5
61-4844    4
61-8942    3
58-1125    3
61-8224    3
61-2495    2
61-4733    2
55-2992    2
58-8701    2
58-1089    2
55-2639    2
61-3846    1
58-9118    1
55-2971    1
Name: count, dtype: int64

Value counts for full_name:
full_name
Laila Mostaf

## Part A - Inconsistency Fixing

### Map values to their canonical forms

We standardize `faculty`, `club`, `city`, and `fee_paid` using explicit mappings. Before mapping, we strip surrounding whitespace and convert text to lowercase so capitalization and extra spaces do not create separate categories.



In [24]:
canonical_maps = {
    'faculty': {
        'met': 'MET',
        'media engineering and technology': 'MET',
        'iet': 'IET',
        'information engineering and technology': 'IET',
        'ems': 'EMS',
        'engineering and materials science': 'EMS',
        'pharmacy': 'Pharmacy',
        'pharma': 'Pharmacy',
        'management': 'Management',
    },
    'club': {
        'robotics': 'Robotics',
        'debate': 'Debate',
        'debating': 'Debate',
        'debate club': 'Debate',
        'music': 'Music',
        'football': 'Football',
        'soccer': 'Football',
        'chess': 'Chess',
        'chess club': 'Chess',
    },
    'city': {
        'cairo': 'Cairo',
        'giza': 'Giza',
        'el giza': 'Giza',
        'alexandria': 'Alexandria',
        'alex': 'Alexandria',
    },
    'fee_paid': {
        'yes': True, 'y': True, 'true': True, '1': True,
        'no': False, 'n': False, 'false': False, '0': False,
    },
}

# Validate every mapping before updating the DataFrame.
mapped_columns = {}
for column, mapping in canonical_maps.items():
    normalized = df[column].astype('string').str.strip().str.lower()
    unrecognized = ~normalized.isin(mapping)
    if unrecognized.any():
        raise ValueError(
            f'Unrecognized or missing values in {column}: '
            f'{df.loc[unrecognized, column].unique().tolist()}'
        )
    mapped_columns[column] = normalized.map(mapping)

for column, mapped in mapped_columns.items():
    df[column] = mapped.astype(bool) if column == 'fee_paid' else mapped

# Check against the required canonical values independently of the mappings.
allowed_values = {
    'faculty': {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'},
    'club': {'Robotics', 'Debate', 'Music', 'Football', 'Chess'},
    'city': {'Cairo', 'Giza', 'Alexandria'},
    'fee_paid': {True, False},
}

for column, allowed in allowed_values.items():
    invalid = ~df[column].isin(allowed)
    assert not invalid.any(), (
        f'Non-canonical or missing values remain in {column}: '
        f'{df.loc[invalid, column].unique().tolist()}'
    )

assert pd.api.types.is_bool_dtype(df['fee_paid']), 'fee_paid must have a boolean dtype'

for column in canonical_maps:
    print(f'Canonical value counts for {column}:')
    print(df[column].value_counts(dropna=False))
    print()
print('fee_paid dtype:', df['fee_paid'].dtype)


Canonical value counts for faculty:
faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

Canonical value counts for club:
club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

Canonical value counts for city:
city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64

Canonical value counts for fee_paid:
fee_paid
True     27
False    12
Name: count, dtype: int64

fee_paid dtype: bool


### Check and standardize names: one spelling per person

We use `student_id` to identify each person. The first cell displays students with multiple original name variants and counts how many spellings remain after removing extra spaces and applying title case. A normalized spelling count above one indicates a spelling difference that needs review.

The following cell fixes capitalization and spacing, verifies that every student has exactly one non-missing name, and displays the resulting names. It stops for review if different spellings remain for the same student ID, rather than choosing a spelling arbitrarily.


In [25]:
# Inspect name variants without changing the DataFrame.
normalized_names = (
    df['full_name'].astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)
name_check = (
    df.assign(normalized_name=normalized_names)
    .groupby('student_id', dropna=False)
    .agg(
        original_variants=('full_name', lambda names: names.unique().tolist()),
        original_spelling_count=('full_name', lambda names: names.nunique(dropna=False)),
        normalized_variants=('normalized_name', lambda names: names.unique().tolist()),
        normalized_spelling_count=('normalized_name', lambda names: names.nunique(dropna=False)),
    )
)
name_variations = name_check[name_check['original_spelling_count'] > 1]
print(f'{len(name_variations)} students have multiple original name variants:')
display(name_variations)


7 students have multiple original name variants:


,original_variants,original_spelling_count,normalized_variants,normalized_spelling_count
student_id,,,,
55-2992,"[mohamed adel, Mohamed Adel]",2,[Mohamed Adel],1
55-9245,"[ALI IBRAHIM, ali ibrahim, Ali Ibrahim]",3,[Ali Ibrahim],1
58-8701,"[Farida Ibrahim, Farida Ibrahim ]",2,[Farida Ibrahim],1
61-2495,"[ahmed adel, AHMED ADEL]",2,[Ahmed Adel],1
61-4733,"[OMAR ADEL, Omar Adel ]",2,[Omar Adel],1
61-4844,"[ Mohamed Adel , MOHAMED ADEL, Mohamed Adel]",3,[Mohamed Adel],1
61-7344,"[Laila Mostafa, Laila Mostafa , laila mostafa]",3,[Laila Mostafa],1


In [26]:
# Standardize spacing and capitalization in each name.
clean_names = (
    df['full_name'].astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)
assert df['student_id'].notna().all(), 'Missing student IDs need review'
assert clean_names.notna().all() and clean_names.ne('').all(), 'Missing or empty names need review'

# Check for remaining spelling differences before updating the DataFrame.
name_counts = clean_names.groupby(df['student_id']).nunique(dropna=False)
conflicting_students = name_counts[name_counts > 1].index.tolist()
assert not conflicting_students, (
    f'Different name spellings still need review for student IDs: {conflicting_students}'
)

df['full_name'] = clean_names
assert df.groupby('student_id')['full_name'].nunique(dropna=False).eq(1).all()

print('Verified: every student ID has exactly one name spelling.')
display(
    df[['student_id', 'full_name']]
    .drop_duplicates()
    .sort_values('student_id')
    .reset_index(drop=True)
)


Verified: every student ID has exactly one name spelling.


,student_id,full_name
0,55-2639,Youssef Fathy
1,55-2971,Salma Ibrahim
2,55-2992,Mohamed Adel
3,55-9245,Ali Ibrahim
4,58-1089,Karim Kamel
5,58-1125,Ziad Saleh
6,58-8701,Farida Ibrahim
7,58-9118,Mariam Kamel
8,61-2495,Ahmed Adel
9,61-3846,Ahmed Zaki


### Normalize email addresses

The email column contains uppercase variants and surrounding spaces. For this dataset, we standardize the addresses to lowercase and remove spaces at the edges before checking for duplicates, so these formatting differences do not hide otherwise identical rows.




In [27]:
clean_emails = df['email'].astype('string').str.strip().str.lower()
assert clean_emails.notna().all() and clean_emails.ne('').all(), (
    'Missing or empty emails need review'
)

df['email'] = clean_emails
display(df[['student_id', 'email']].head())


,student_id,email
0,61-8942,hana.zaki@student.guc.edu.eg
1,61-3846,ahmed.zaki@student.guc.edu.eg
2,61-4844,mohamed.adel@student.guc.edu.eg
3,61-4844,mohamed.adel@student.guc.edu.eg
4,61-7344,laila.mostafa@student.guc.edu.eg


### Standardize signup dates

`signed_up_at` contains 25 dates in `YYYY-MM-DD HH:MM` format and 14 in `DD/MM/YYYY HH:MM` format. The slash dates are unambiguously day-first: the first component ranges from 13 to 18 (too large to be a month), while the second is always 09 (September). We therefore parse the mixed formats with `dayfirst=True`, converting the column to datetime while preserving the time. Invalid dates raise an error rather than silently becoming missing values.



In [28]:
df['signed_up_at'] = pd.to_datetime(
    df['signed_up_at'], format='mixed', dayfirst=True, errors='raise'
)
assert df['signed_up_at'].notna().all(), 'Missing signup dates need review'

print('signed_up_at dtype:', df['signed_up_at'].dtype)
display(df[['signup_id', 'signed_up_at']].head())


signed_up_at dtype: datetime64[ns]


,signup_id,signed_up_at
0,S015,2026-09-15 13:39:00
1,S010,2026-09-15 03:12:00
2,S008,2026-09-14 23:32:00
3,S035,2026-09-17 11:15:00
4,S004,2026-09-13 22:36:00


### Count, inspect, and remove exact duplicates

An exact duplicate matches another row in every column after the standardization above. Count the extra copies that will be removed, and display all rows in duplicate groups (including the first copy) for inspection. Then keep one copy of each distinct row.


In [29]:
exact_duplicate_count = df.duplicated().sum()
print(f'Exact duplicate rows to remove: {exact_duplicate_count}')
display(df[df.duplicated(keep=False)])

df = df.drop_duplicates().reset_index(drop=True)


Exact duplicate rows to remove: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False


### Keep the latest signup for each student and club

After removing exact duplicates, inspect every remaining group where the same `student_id` signed up for the same canonical `club` more than once. Sort by the datetime `signed_up_at` and keep the latest row per pair, then report how many rows were removed. If timestamps tie, keep the row that appeared last in the current dataset; stable sorting preserves that order. A student can still sign up for different clubs.




In [30]:
signup_key = ['student_id', 'club']
repeated_signups = df[df.duplicated(subset=signup_key, keep=False)]
display(repeated_signups.sort_values(signup_key + ['signed_up_at']))

rows_before = len(df)
df = (
    df.sort_values('signed_up_at', kind='stable')
    .drop_duplicates(subset=signup_key, keep='last')
    .reset_index(drop=True)
)
print(f'Repeated student-club signup rows removed: {rows_before - len(df)}')


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
32,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
24,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
35,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


Repeated student-club signup rows removed: 4


### Assert student–club uniqueness




In [31]:
assert not df.duplicated(subset=['student_id', 'club']).any(), (
    'Duplicate student-club signups remain'
)
print(f'Verified: all {len(df)} student-club pairs are unique.')


Verified: all 32 student-club pairs are unique.


### Save the cleaned dataset


In [ ]:
df.to_csv('club_signups_clean.csv', index=False)
